# Eksperimen Topic Modeling FAQ — Perbandingan 3 Model Embedding + Hyperparameter Search

Notebook riset end-to-end buat sistem FAQ clustering dengan continuous retraining:
load data -> EDA ringkas -> preprocessing -> generate embedding (3 model
dibandingkan) -> hyperparameter search ekstensif -> evaluasi -> topic labeling
pakai Gemini -> visualisasi -> bundling semua output ke ZIP.

Notebook ini murni buat EKSPERIMEN/RISET (bukan pipeline production), jadi
kolom `category` sengaja dipertahankan sepanjang notebook buat validasi
(NMI/ARI/Purity) — tapi TIDAK PERNAH ikut jadi fitur embedding/clustering.

**Cara pakai:** run semua cell dari atas ke bawah (`Runtime > Run all`).
Satu-satunya yang WAJIB diisi manual sebelum run: `GEMINI_API_KEY` di cell
konfigurasi Section 0.

# Section 0 — Setup

In [ ]:
!pip install -q pandas numpy matplotlib seaborn wordcloud sentence-transformers bertopic umap-learn hdbscan scikit-learn gensim google-generativeai

In [ ]:
import os
import re
import time
import random
import shutil
import unicodedata
import zipfile
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sentence_transformers import SentenceTransformer
import umap
import hdbscan
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from bertopic import BERTopic

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

import google.generativeai as genai

warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# KONFIGURASI UTAMA — variabel yang paling sering diubah ada di sini
# ============================================================

GEMINI_API_KEY = "PASTE_API_KEY_ANDA_DI_SINI"  # hardcode, ini notebook eksperimen
GEMINI_MODEL_NAME = "gemini-1.5-flash"  # ganti kalau model ini udah deprecated pas kamu run notebook ini
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 3 model embedding yang dibandingkan, sengaja beda generasi & ukuran:
#   - minilm    : all-MiniLM-L6-v2       (384 dim, 2021, kecil & cepat -> baseline ringan)
#   - mpnet     : all-mpnet-base-v2      (768 dim, 2021, besar & akurat -> baseline "mahal tapi bagus")
#   - bge_small : BAAI/bge-small-en-v1.5 (384 dim, 2023, model modern -> apa model baru yang
#                 kecil bisa ngalahin model lama yang lebih besar?)
# Ketiganya bisa langsung di-load lewat sentence-transformers tanpa token/API key tambahan.
# Mau nambah model ke-4/ke-5? Tinggal tambah 1 baris di dict ini, semua loop di bawah otomatis ikut.
EMBEDDING_MODELS = {
    "minilm": "all-MiniLM-L6-v2",
    "mpnet": "all-mpnet-base-v2",
    "bge_small": "BAAI/bge-small-en-v1.5",
}

N_COMBINATIONS_PER_MODEL = 60  # random search budget PER model embedding (naikin kalau mau lebih ekstensif)
COHERENCE_SAMPLE_SIZE = 3000   # subsample dokumen buat hitung coherence (turunin kalau kelamaan dihitung)

OUTPUT_DIR = "/content/outputs"
DATA_PATH = "/content/faq_training_set.csv"

if GEMINI_API_KEY == "PASTE_API_KEY_ANDA_DI_SINI":
    print("PERINGATAN: GEMINI_API_KEY belum diisi.")
    print("Section 0-6 tetap bisa jalan, tapi Section 7 (topic labeling) bakal error")
    print("kalau key ini belum diganti sebelum sampai sana.")

In [ ]:
# ============================================================
# Estimasi kasar waktu eksekusi Section 5 (hyperparameter search)
# ============================================================
_DETIK_PER_KOMBINASI_ASUMSI = 18  # asumsi rata-rata CPU Colab standar, actual bisa beda-beda
_total_kombinasi = N_COMBINATIONS_PER_MODEL * len(EMBEDDING_MODELS)
_estimasi_menit = _total_kombinasi * _DETIK_PER_KOMBINASI_ASUMSI / 60

print("=" * 60)
print(f"Total kombinasi yang bakal dicoba : {_total_kombinasi} ({N_COMBINATIONS_PER_MODEL} x {len(EMBEDDING_MODELS)} model)")
print(f"Estimasi waktu Section 5 (search) : ~{_estimasi_menit:.0f} menit (kasar, CPU-only, gak butuh GPU)")
print("Plus estimasi ~10-15 menit buat generate embedding (Section 4) + fit model final & labeling (Section 7).")
print("Waktu aktual tergantung ukuran data setelah cleaning & spek CPU Colab kamu.")
print("Mau lebih cepat? Turunin N_COMBINATIONS_PER_MODEL di cell konfigurasi di atas.")
print("=" * 60)

In [ ]:
if not os.path.exists(DATA_PATH):
    print("File faq_training_set.csv belum ketemu di /content/, silakan upload manual...")
    from google.colab import files
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    if os.path.abspath(uploaded_name) != os.path.abspath(DATA_PATH):
        shutil.move(uploaded_name, DATA_PATH)
    print(f"File tersimpan di {DATA_PATH}")
else:
    print(f"File sudah ada di {DATA_PATH}, lanjut ke Section 1.")

# Section 1 — Load Data

In [ ]:
def load_raw_data(path: str) -> pd.DataFrame:
    return pd.read_csv(path)


df_raw = load_raw_data(DATA_PATH)

print("Shape:", df_raw.shape)
display(df_raw.head())
print("\nDistribusi kategori:")
print(df_raw["category"].value_counts())

# Section 2 — EDA (Ringkas)

EDA mendalam udah pernah dilakukan di notebook lain (`Eksperimen_Aiman_EDA.ipynb`),
jadi di sini cukup versi ringkas buat konfirmasi karakteristik data sebelum
masuk preprocessing.

In [ ]:
def plot_text_length_distribution(df: pd.DataFrame, text_col: str = "utterance") -> None:
    n_words = df[text_col].str.split().str.len()
    n_chars = df[text_col].str.len()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(n_words, bins=30, color="#4C72B0", edgecolor="white")
    axes[0].set_title("Distribusi Jumlah Kata")
    axes[1].hist(n_chars, bins=30, color="#DD8452", edgecolor="white")
    axes[1].set_title("Distribusi Jumlah Karakter")
    plt.tight_layout()
    plt.show()

    print(pd.DataFrame({"n_words": n_words, "n_chars": n_chars}).describe())


plot_text_length_distribution(df_raw)

In [ ]:
def check_missing_and_duplicates(df: pd.DataFrame, text_col: str = "utterance") -> None:
    print("Missing value per kolom:")
    print(df.isna().sum())

    n_exact_dup = df[text_col].duplicated().sum()
    print(f"\nExact duplicate: {n_exact_dup} ({n_exact_dup / len(df) * 100:.2f}%)")

    normalized = df[text_col].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
    n_near_dup = normalized.duplicated().sum()
    print(f"Near-duplicate (lowercase+strip): {n_near_dup} ({n_near_dup / len(df) * 100:.2f}%)")


check_missing_and_duplicates(df_raw)

In [ ]:
def plot_wordcloud(df: pd.DataFrame, text_col: str = "utterance") -> None:
    text_gabungan = " ".join(df[text_col].astype(str).tolist())
    wc = WordCloud(width=900, height=450, background_color="white", colormap="viridis").generate(text_gabungan)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title("WordCloud Seluruh Pertanyaan")
    plt.show()


plot_wordcloud(df_raw)

# Section 3 — Preprocessing

Semua langkah diterapkan di SATU dataframe (`df_raw` -> `df_clean`). Kolom
`category` (dan kolom lain) ikut ke-filter otomatis karena row yang sama —
TIDAK di-drop, cuma TIDAK PERNAH dipakai sebagai fitur.

Cleaning cuma ringan (unicode normalize + rapikan whitespace) — TIDAK
stopword removal/stemming/lowercase paksa/hapus tanda baca, karena bakal
ngerusak konteks yang dibutuhkan sentence-transformer buat nangkep makna
semantik.

In [ ]:
def filter_short_utterances(df: pd.DataFrame, min_words: int = 3, text_col: str = "utterance") -> pd.DataFrame:
    before = len(df)
    n_words = df[text_col].str.split().str.len()
    df_filtered = df.loc[n_words >= min_words].copy()
    print(f"[filter_short_utterances] {before} -> {len(df_filtered)} baris (buang {before - len(df_filtered)} baris < {min_words} kata)")
    return df_filtered


def drop_near_duplicates(df: pd.DataFrame, text_col: str = "utterance") -> pd.DataFrame:
    before = len(df)
    normalized_key = df[text_col].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
    df_deduped = df.loc[~normalized_key.duplicated(keep="first")].copy()
    print(f"[drop_near_duplicates] {before} -> {len(df_deduped)} baris (buang {before - len(df_deduped)} near-duplicate)")
    return df_deduped


def light_clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"\s+", " ", text).strip()


def apply_light_cleaning(df: pd.DataFrame, text_col: str = "utterance") -> pd.DataFrame:
    df = df.copy()
    df[text_col] = df[text_col].apply(light_clean_text)
    print(f"[apply_light_cleaning] Unicode normalize (NFKC) + rapikan whitespace diterapkan ke {len(df)} baris")
    return df


def preprocess_pipeline(df_raw: pd.DataFrame) -> pd.DataFrame:
    print(f"Baris awal: {len(df_raw)}")
    df = filter_short_utterances(df_raw)
    df = drop_near_duplicates(df)
    df = apply_light_cleaning(df)
    df = df.reset_index(drop=True)
    print(f"Baris akhir: {len(df)}")
    return df


df_clean = preprocess_pipeline(df_raw)
df_clean[["utterance", "category"]].head()

# Section 4 — Generate Embedding (3 Model Dibandingkan)

Generate embedding dari `df_clean['utterance']` pakai 3 model sentence-transformer
berbeda, disimpan in-memory (dict numpy array, bukan file) buat langsung dipakai
di hyperparameter search Section 5.

Kalau runtime Colab kamu pakai GPU (`Runtime > Change runtime type`),
sentence-transformers otomatis pakai GPU buat generate embedding lebih cepat.
UMAP/HDBSCAN di Section 5 tetap CPU-only (gak butuh GPU).

In [ ]:
def generate_embeddings(texts: list[str], model_name: str) -> np.ndarray:
    print(f"Loading model: {model_name} ...")
    model = SentenceTransformer(model_name)
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
    print(f"Selesai. Shape embedding: {embeddings.shape}")
    del model
    return embeddings


utterances = df_clean["utterance"].tolist()

embeddings_by_model = {}
for _key, _model_name in EMBEDDING_MODELS.items():
    embeddings_by_model[_key] = generate_embeddings(utterances, _model_name)

print("\nRingkasan shape embedding:")
for _key, _emb in embeddings_by_model.items():
    print(f"  {_key} ({EMBEDDING_MODELS[_key]}): shape = {_emb.shape}")

# Section 5 — Hyperparameter Search (Random Search, 3 Model Embedding)

Random search (bukan grid penuh — grid penuh kelamaan buat Colab) dengan budget
`N_COMBINATIONS_PER_MODEL` kombinasi UNIK per model embedding, disampling dari
search space UMAP + HDBSCAN di bawah.

Search space HDBSCAN sengaja ditambah `cluster_selection_method` (eom vs leaf)
biar lebih ekstensif dari versi awal. UMAP `metric` DIFIKSASI ke `"cosine"`
(bukan jadi hyperparameter yang di-random) karena itu praktik standar buat
embedding sentence-transformer (dibandingkan lewat cosine similarity) — kalau
ikut di-random, kombinasi `euclidean` hampir selalu kalah dan cuma buang-buang
budget kombinasi yang bisa dipakai buat eksplorasi parameter lain.

Untuk tiap kombinasi: fit UMAP -> fit HDBSCAN -> hitung NMI/ARI/Purity
(exclude outlier) + outlier_ratio + n_topics + coherence (c_v, proxy top-kata
per cluster — BUKAN c-TF-IDF BERTopic penuh, itu baru di Section 7 buat speed).

In [ ]:
UMAP_SEARCH_SPACE = {
    "n_neighbors": [10, 15, 20, 30, 50],
    "n_components": [5, 10, 15],
    "min_dist": [0.0, 0.05, 0.1],
}

HDBSCAN_SEARCH_SPACE = {
    "min_cluster_size": [15, 30, 50, 75, 100],
    "min_samples": [None, 5, 10],
    "cluster_selection_method": ["eom", "leaf"],
}

_umap_space_size = len(UMAP_SEARCH_SPACE["n_neighbors"]) * len(UMAP_SEARCH_SPACE["n_components"]) * len(UMAP_SEARCH_SPACE["min_dist"])
_hdbscan_space_size = len(HDBSCAN_SEARCH_SPACE["min_cluster_size"]) * len(HDBSCAN_SEARCH_SPACE["min_samples"]) * len(HDBSCAN_SEARCH_SPACE["cluster_selection_method"])
_total_space_size = _umap_space_size * _hdbscan_space_size

print(f"Total ruang kombinasi: {_total_space_size} kemungkinan (UMAP: {_umap_space_size} x HDBSCAN: {_hdbscan_space_size})")
print(f"Random search sample: {N_COMBINATIONS_PER_MODEL} kombinasi/model ({N_COMBINATIONS_PER_MODEL / _total_space_size * 100:.1f}% dari ruang kombinasi)")

In [ ]:
def sample_hyperparameter_combinations(n: int, rng: random.Random) -> list[dict]:
    """Random search: sampling n kombinasi UNIK dari search space (skip duplikat)."""
    seen = set()
    combos = []
    max_tries = n * 20
    tries = 0
    while len(combos) < n and tries < max_tries:
        tries += 1
        combo = {
            "n_neighbors": rng.choice(UMAP_SEARCH_SPACE["n_neighbors"]),
            "n_components": rng.choice(UMAP_SEARCH_SPACE["n_components"]),
            "min_dist": rng.choice(UMAP_SEARCH_SPACE["min_dist"]),
            "min_cluster_size": rng.choice(HDBSCAN_SEARCH_SPACE["min_cluster_size"]),
            "min_samples": rng.choice(HDBSCAN_SEARCH_SPACE["min_samples"]),
            "cluster_selection_method": rng.choice(HDBSCAN_SEARCH_SPACE["cluster_selection_method"]),
        }
        key = tuple(combo.values())
        if key in seen:
            continue
        seen.add(key)
        combos.append(combo)
    return combos


def purity_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    contingency = pd.crosstab(y_pred, y_true)
    return contingency.max(axis=1).sum() / contingency.values.sum()


_TOKEN_PATTERN = re.compile(r"[a-zA-Z']+")


def tokenize_simple(text: str) -> list[str]:
    return [tok.lower() for tok in _TOKEN_PATTERN.findall(text)]


def get_top_words_per_cluster(texts: list[str], cluster_labels: np.ndarray, top_n: int = 10) -> dict[int, list[str]]:
    """Kata paling sering muncul per cluster (proxy topik simpel buat speed di tahap
    search ini -- c-TF-IDF resmi BERTopic baru dipakai di Section 7)."""
    top_words = {}
    for cluster_id in sorted(set(cluster_labels)):
        if cluster_id == -1:
            continue
        cluster_texts = [texts[i] for i in range(len(texts)) if cluster_labels[i] == cluster_id]
        tokens = [tok for text in cluster_texts for tok in tokenize_simple(text)]
        counts = Counter(tokens)
        top_words[cluster_id] = [word for word, _ in counts.most_common(top_n)]
    return top_words


def compute_coherence(top_words_per_cluster: dict[int, list[str]], coherence_texts: list[list[str]], dictionary: Dictionary) -> float:
    topics = [words for words in top_words_per_cluster.values() if len(words) > 0]
    if len(topics) < 2:
        return float("nan")
    cm = CoherenceModel(topics=topics, texts=coherence_texts, dictionary=dictionary, coherence="c_v")
    return cm.get_coherence()

In [ ]:
print("Menyiapkan corpus referensi buat coherence (c_v)...")
_rng_coherence = random.Random(RANDOM_SEED)
_sample_idx = _rng_coherence.sample(range(len(df_clean)), k=min(COHERENCE_SAMPLE_SIZE, len(df_clean)))
coherence_texts = [tokenize_simple(df_clean["utterance"].iloc[i]) for i in _sample_idx]
coherence_dictionary = Dictionary(coherence_texts)
print(f"Corpus referensi coherence: {len(coherence_texts)} dokumen (subsample dari {len(df_clean)})")

In [ ]:
def evaluate_combination(
    embedding_key: str,
    embeddings: np.ndarray,
    texts: list[str],
    categories: np.ndarray,
    combo: dict,
    coherence_texts: list[list[str]],
    coherence_dictionary: Dictionary,
    seed: int,
) -> dict:
    t0 = time.time()

    reducer = umap.UMAP(
        n_neighbors=combo["n_neighbors"],
        n_components=combo["n_components"],
        min_dist=combo["min_dist"],
        metric="cosine",
        random_state=seed,
    )
    reduced = reducer.fit_transform(embeddings)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=combo["min_cluster_size"],
        min_samples=combo["min_samples"],
        cluster_selection_method=combo["cluster_selection_method"],
        metric="euclidean",
    )
    cluster_labels = clusterer.fit_predict(reduced)

    is_outlier = cluster_labels == -1
    outlier_ratio = is_outlier.sum() / len(cluster_labels)
    n_topics = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)

    if n_topics >= 2:
        mask = ~is_outlier
        nmi = normalized_mutual_info_score(categories[mask], cluster_labels[mask])
        ari = adjusted_rand_score(categories[mask], cluster_labels[mask])
        purity = purity_score(categories[mask], cluster_labels[mask])
        top_words = get_top_words_per_cluster(texts, cluster_labels)
        coherence = compute_coherence(top_words, coherence_texts, coherence_dictionary)
    else:
        nmi = ari = purity = coherence = float("nan")

    waktu_eksekusi = time.time() - t0

    return {
        "embedding_model": embedding_key,
        **combo,
        "nmi": nmi,
        "ari": ari,
        "purity": purity,
        "outlier_ratio": outlier_ratio,
        "n_topics": n_topics,
        "coherence_cv": coherence,
        "waktu_eksekusi": waktu_eksekusi,
    }

In [ ]:
categories = df_clean["category"].values
texts_all = df_clean["utterance"].tolist()

leaderboard_rows = []
rng_search = random.Random(RANDOM_SEED)
_search_start = time.time()

for embedding_key, embeddings in embeddings_by_model.items():
    print(f"\n{'=' * 60}\nMulai random search untuk embedding: {embedding_key} ({EMBEDDING_MODELS[embedding_key]})\n{'=' * 60}")
    combinations = sample_hyperparameter_combinations(N_COMBINATIONS_PER_MODEL, rng_search)

    for i, combo in enumerate(combinations, start=1):
        result = evaluate_combination(
            embedding_key=embedding_key,
            embeddings=embeddings,
            texts=texts_all,
            categories=categories,
            combo=combo,
            coherence_texts=coherence_texts,
            coherence_dictionary=coherence_dictionary,
            seed=RANDOM_SEED,
        )
        leaderboard_rows.append(result)

        if i % 10 == 0 or i == len(combinations):
            print(f"[{embedding_key}] Kombinasi {i}/{len(combinations)} selesai "
                  f"(NMI={result['nmi']:.3f}, outlier={result['outlier_ratio']:.2%}, waktu={result['waktu_eksekusi']:.1f}s)")

leaderboard_df = pd.DataFrame(leaderboard_rows)
print(f"\nTotal kombinasi selesai: {len(leaderboard_df)} dalam {(time.time() - _search_start) / 60:.1f} menit")
leaderboard_df.head()

# Section 6 — Leaderboard & Pemilihan Model Terbaik

In [ ]:
def compute_composite_score(df: pd.DataFrame) -> pd.Series:
    score = 0.4 * df["nmi"].fillna(0) + 0.3 * df["ari"].fillna(0) + 0.3 * (1 - df["outlier_ratio"])
    # kombinasi yang gagal bikin >=2 topik dianggap gagal total, gak boleh menang cuma
    # gara-gara outlier_ratio-nya kebetulan rendah
    return score.where(df["n_topics"] >= 2, other=0.0)


leaderboard_df["composite_score"] = compute_composite_score(leaderboard_df)

leaderboard_sorted = leaderboard_df.sort_values("composite_score", ascending=False)
print("Top 10 kombinasi keseluruhan:")
display(leaderboard_sorted.head(10).round(4))

In [ ]:
comparison_df = (
    leaderboard_df.groupby("embedding_model")[["composite_score", "nmi", "ari", "purity", "outlier_ratio", "coherence_cv"]]
    .mean()
    .round(4)
    .sort_values("composite_score", ascending=False)
)
print("Perbandingan rata-rata metrik per model embedding (minilm vs mpnet vs bge_small):")
print("-> menjawab: apakah dimensi lebih besar / model lebih modern benar-benar lebih baik buat kasus ini?")
display(comparison_df)

In [ ]:
best_idx = leaderboard_df["composite_score"].idxmax()
BEST_CONFIG = leaderboard_rows[best_idx]  # ambil dari list asli (native Python types, None tetap None)

print("BEST_CONFIG terpilih:")
for k, v in BEST_CONFIG.items():
    print(f"  {k}: {v}")

# Section 7 — Fit Model Final + Topic Labeling via Gemini

Fit ulang BERTopic PENUH (bukan UMAP+HDBSCAN mentah lagi) pakai `BEST_CONFIG`
dari Section 6, supaya dapat representasi c-TF-IDF resmi dari BERTopic.

In [ ]:
best_embedding_key = BEST_CONFIG["embedding_model"]
best_embeddings = embeddings_by_model[best_embedding_key]

final_umap_model = umap.UMAP(
    n_neighbors=BEST_CONFIG["n_neighbors"],
    n_components=BEST_CONFIG["n_components"],
    min_dist=BEST_CONFIG["min_dist"],
    metric="cosine",
    random_state=RANDOM_SEED,
)

final_hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=BEST_CONFIG["min_cluster_size"],
    min_samples=BEST_CONFIG["min_samples"],
    cluster_selection_method=BEST_CONFIG["cluster_selection_method"],
    metric="euclidean",
    prediction_data=True,  # biar bisa .transform() dokumen baru nanti (continuous retraining)
)

embedding_model_final = SentenceTransformer(EMBEDDING_MODELS[best_embedding_key])

topic_model = BERTopic(
    embedding_model=embedding_model_final,
    umap_model=final_umap_model,
    hdbscan_model=final_hdbscan_model,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(df_clean["utterance"].tolist(), embeddings=best_embeddings)
df_clean["topic"] = topics

n_topics_final = len(set(topics)) - (1 if -1 in topics else 0)
print(f"\nModel embedding terpilih : {best_embedding_key} ({EMBEDDING_MODELS[best_embedding_key]})")
print(f"Jumlah topik ditemukan   : {n_topics_final}")
topic_model.get_topic_info().head(15)

## Topic Labeling via Gemini

Gemini dipanggil HANYA sekali PER TOPIK (bukan per dokumen), jadi jumlah API
call = jumlah topik, bukan ribuan.

In [ ]:
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel(GEMINI_MODEL_NAME)


def build_labeling_prompt(top_words: list[str], sample_docs: list[str]) -> str:
    words_str = ", ".join(top_words)
    docs_str = "\n".join(f"- {doc}" for doc in sample_docs)
    return f"""Kamu ahli topic modeling. Berdasarkan info topik berikut, kasih SATU label
singkat yang human-readable buat topik ini (contoh: "Payment & Billing Issues").

Kata kunci teratas (c-TF-IDF): {words_str}

Contoh dokumen paling representatif:
{docs_str}

Jawab HANYA dengan label singkatnya aja (maks 5 kata), tanpa penjelasan tambahan."""


def label_topic_with_gemini(top_words: list[str], sample_docs: list[str]) -> str:
    prompt = build_labeling_prompt(top_words, sample_docs)
    response = gemini_model.generate_content(prompt)
    return response.text.strip()

In [ ]:
topic_labels_rows = []
topic_ids = [t for t in topic_model.get_topic_info()["Topic"] if t != -1]

print(f"Memanggil Gemini API untuk {len(topic_ids)} topik (1 call per topik)...")
for i, topic_id in enumerate(topic_ids, start=1):
    top_words = [word for word, _ in topic_model.get_topic(topic_id)[:10]]
    sample_docs = topic_model.get_representative_docs(topic_id)[:3]

    gemini_label = label_topic_with_gemini(top_words, sample_docs)

    topic_labels_rows.append({
        "topic_id": topic_id,
        "top_words": ", ".join(top_words),
        "sample_docs": " | ".join(sample_docs),
        "gemini_label": gemini_label,
    })
    print(f"Topik {i}/{len(topic_ids)} (id={topic_id}) -> {gemini_label}")
    time.sleep(1)  # jaga-jaga rate limit API gratis

topic_labels_df = pd.DataFrame(topic_labels_rows)
topic_labels_df

# Section 8 — Visualisasi Final

In [ ]:
def plot_intertopic_map(embeddings: np.ndarray, cluster_labels: np.ndarray, title: str, seed: int) -> plt.Figure:
    reducer_2d = umap.UMAP(n_components=2, metric="cosine", random_state=seed)
    coords_2d = reducer_2d.fit_transform(embeddings)

    fig, ax = plt.subplots(figsize=(10, 8))
    is_outlier = cluster_labels == -1
    scatter = ax.scatter(
        coords_2d[~is_outlier, 0], coords_2d[~is_outlier, 1],
        c=cluster_labels[~is_outlier], cmap="tab20", s=8, alpha=0.7,
    )
    ax.scatter(coords_2d[is_outlier, 0], coords_2d[is_outlier, 1], c="lightgray", s=5, alpha=0.3, label="outlier")
    ax.set_title(title)
    ax.legend()
    plt.colorbar(scatter, ax=ax, label="Cluster ID")
    return fig


fig_intertopic = plot_intertopic_map(
    best_embeddings, np.array(topics), f"Intertopic Distance Map ({best_embedding_key})", RANDOM_SEED
)
plt.show()

In [ ]:
def plot_top_words_per_topic(topic_model: BERTopic, n_topics: int = 5, n_words: int = 10) -> plt.Figure:
    topic_info = topic_model.get_topic_info()
    top_topic_ids = topic_info[topic_info["Topic"] != -1].head(n_topics)["Topic"].tolist()

    fig, axes = plt.subplots(1, len(top_topic_ids), figsize=(5 * len(top_topic_ids), 4))
    if len(top_topic_ids) == 1:
        axes = [axes]

    for ax, topic_id in zip(axes, top_topic_ids):
        words_scores = topic_model.get_topic(topic_id)[:n_words]
        words = [w for w, _ in words_scores][::-1]
        scores = [s for _, s in words_scores][::-1]
        ax.barh(words, scores, color="#4C72B0")
        ax.set_title(f"Topik {topic_id}")

    plt.tight_layout()
    return fig


fig_top_words = plot_top_words_per_topic(topic_model)
plt.show()

In [ ]:
def plot_composite_score_comparison(leaderboard_df: pd.DataFrame) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=leaderboard_df, x="embedding_model", y="composite_score", ax=ax, hue="embedding_model", palette="Set2", legend=False)
    ax.set_title("Distribusi Composite Score per Model Embedding (semua kombinasi)")
    return fig


fig_score_comparison = plot_composite_score_comparison(leaderboard_df)
plt.show()

In [ ]:
def plot_topic_size_distribution(topic_model: BERTopic) -> plt.Figure:
    topic_info = topic_model.get_topic_info()
    topic_info = topic_info[topic_info["Topic"] != -1].sort_values("Count", ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(range(len(topic_info)), topic_info["Count"], color="#55A868")
    ax.set_xlabel("Topik (diurutkan berdasarkan ukuran)")
    ax.set_ylabel("Jumlah dokumen")
    ax.set_title("Distribusi Ukuran Topik")
    return fig


fig_topic_sizes = plot_topic_size_distribution(topic_model)
plt.show()

In [ ]:
def plot_contingency_heatmap(cluster_labels: np.ndarray, categories: np.ndarray) -> plt.Figure:
    contingency = pd.crosstab(cluster_labels, categories)
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(contingency, cmap="YlOrRd", ax=ax, cbar_kws={"label": "Jumlah dokumen"})
    ax.set_title("Heatmap Kontingensi: Cluster (hasil) vs Category (asli)")
    ax.set_xlabel("Category (asli)")
    ax.set_ylabel("Cluster (hasil)")
    return fig


fig_contingency = plot_contingency_heatmap(np.array(topics), df_clean["category"].values)
plt.show()

# Section 9 — Simpan & Bundling Semua Output ke ZIP

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

leaderboard_df.to_csv(os.path.join(OUTPUT_DIR, "leaderboard_df.csv"), index=False)
topic_labels_df.to_csv(os.path.join(OUTPUT_DIR, "topic_labels_df.csv"), index=False)
print("leaderboard_df.csv & topic_labels_df.csv tersimpan")

figures_to_save = {
    "intertopic_distance_map": fig_intertopic,
    "top_words_per_topic": fig_top_words,
    "composite_score_comparison": fig_score_comparison,
    "topic_size_distribution": fig_topic_sizes,
    "contingency_heatmap": fig_contingency,
}
for name, fig in figures_to_save.items():
    fig.savefig(os.path.join(OUTPUT_DIR, f"{name}.png"), dpi=150, bbox_inches="tight")
print(f"{len(figures_to_save)} gambar tersimpan (.png)")

model_save_path = os.path.join(OUTPUT_DIR, "bertopic_model_best")
topic_model.save(model_save_path, serialization="safetensors", save_ctfidf=True, save_embedding_model=True)
print(f"Model BERTopic tersimpan di {model_save_path}")

In [ ]:
ringkasan_lines = [
    "RINGKASAN EKSPERIMEN -- TOPIC MODELING FAQ CLUSTERING",
    "=" * 60,
    "",
    "BEST_CONFIG:",
]
for k, v in BEST_CONFIG.items():
    ringkasan_lines.append(f"  {k}: {v}")

ringkasan_lines += [
    "",
    "PERBANDINGAN ANTAR MODEL EMBEDDING (rata-rata seluruh kombinasi):",
    comparison_df.to_string(),
    "",
    f"Total kombinasi dicoba : {len(leaderboard_df)} ({N_COMBINATIONS_PER_MODEL} per model x {len(EMBEDDING_MODELS)} model)",
    f"Jumlah topik final     : {n_topics_final}",
]

ringkasan_text = "\n".join(ringkasan_lines)
with open(os.path.join(OUTPUT_DIR, "ringkasan_eksperimen.txt"), "w", encoding="utf-8") as f:
    f.write(ringkasan_text)

print(ringkasan_text)

In [ ]:
zip_path = "/content/faq_topic_modeling_experiment_results.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for filename in files:
            filepath = os.path.join(root, filename)
            arcname = os.path.relpath(filepath, OUTPUT_DIR)
            zf.write(filepath, arcname)

print(f"Zip selesai: {zip_path} ({os.path.getsize(zip_path) / 1024:.1f} KB)")

## Download Hasil

Zip otomatis ke-download begitu cell di bawah ini selesai jalan.

In [ ]:
from google.colab import files
files.download(zip_path)